In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 1. GEMINI RAG

In [ ]:
import pandas as pd
from google import genai

def trainingDataSetSampling():

    # =========================
    # 1. CSV 로드
    # =========================
    input_path = '/KOR_hatespeech_all_train.csv'
    df = pd.read_csv(input_path)
    df["comments"] = df["comments"].astype(str).str.slice(0, 50) # 코멘트 길이 50자로 자르기

    # =========================
    # 2. 설정
    # =========================
    LABEL_COL = "label"      # ← 레이블 컬럼명 (필요시 수정)
    MAX_PER_LABEL = 3000 # todo: change 5000, 10000
    RANDOM_SEED = 42

    # =========================
    # 3. 레이블별 최대 1만개 샘플링
    # =========================
    df_sampled = (
        df
        .groupby(LABEL_COL, group_keys=False)
        .apply(lambda x: x.sample(
            n=min(len(x), MAX_PER_LABEL),
            random_state=RANDOM_SEED
        ))
    )

    # =========================
    # 4. 결과 확인
    # =========================
    print(len(df_sampled))
    print(df_sampled[LABEL_COL].value_counts())

    # =========================
    # 5. 저장
    # =========================
    output_path = '/KOR_hatespeech_all_train_sampling.csv'
    df_sampled.to_csv(output_path, index=False)

    print(f"Saved to: {output_path}")

client = genai.Client(api_key="YOUR_API_KEY")

trainingDataSetSampling()

train_dataset = client.files.upload(file="/KOR_hatespeech_all_train.csv")

response = client.models.generate_content(
    model="gemini-2.5-flash", contents=[
        """
        너의 업무는 내가 전달하는 데이터를 학습하는거야.
        각각의 행에는 코멘트와 레이블이 포함되어 있어. 레이블의 카테고리는 숫자로 표현되고,  0번부터 7번에 해당하는 혐오표현의 경우 다중으로 라벨링 될 수 있어.
        코멘트를 내용과 레이블의 카테고리를 학습하는거야. 카테고리의 종류는 아래와 같아.

        ### 카테고리 정의
        0 출신차별(origin): 출신 지역, 고향 등을 모욕 또는 비난하거나 배제하고 차별을 정당화하는 경우. 출신 국가, 지역, 국적, 이민 상태에 기반한 멸칭 등을 포함한 혐오 발언
        1 외모차별(physical): 신체적 특징, 장애로 조롱하고 비난하거나 이로 인해 개인의 가치나 능력을 낮게 평가하고 비난하는 발언
        2 정치성향차별(political): 특정 정당 지지, 이념적 견해, 정치 성향 등을 이유로 비난하고 이를 정당화하는 발언
        3 혐오욕설(profanitiy): 일반적인 욕설, 특정한 개인에 대한 비속어 형태의 비난이 포함되어 있는 발언
        4 연령차별(age): 나이 또는 세대로 집단을 비하하고 조롱하는 발언
        5 성차별(gender): 성별, 개인의 성 지향성을 기반으로 편견을 가지고 차별하고 비난하는 발언
        6 인종차별(race): 피부색, 인종, 민족적 배경 등을 이유로 개인이나 집단을 모욕하는 경우. 인종, 피부색, 민족 혈통 자체에 기반한 혐오 발언
        7 종교차별(religion): 특정 종교에 대한 편견으로 개인이나 집단을 모욕하는 발언
        8 해당사항없음(not hate speech): 위 기준에 해당되지 않음. 혐오발언 없음.
        """, train_dataset]
)


/tmp/ipython-input-16380165.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


39167
label
0          3000
2          3000
1          3000
3          3000
8          3000
           ... 
1,3,5,7       1
1,2,7         1
2,3,5,6       1
2,6,7         1
4,5,7         1
Name: count, Length: 99, dtype: int64
Saved to: /content/drive/MyDrive/MP_KORHateSpeechDetection/KOR_hatespeech_all_train_sampling.csv


In [ ]:
import ast
import time
import json
import re
from collections import defaultdict, Counter

import pandas as pd
from google import genai


client = genai.Client(api_key="YOUR_API_KEY")


def parse_llm_dict(text):
    text = text.strip()
    if text.startswith("```"):
        text = "\n".join(text.splitlines()[1:-1]).strip()
    return ast.literal_eval(text)


# ============================================================
# ✅ [ADD 1] Train(knowledge base) 로드 + Retriever 준비
# ============================================================
TRAIN_PATH = "/KOR_hatespeech_all_train.csv"

# 구분자 ; 가능성 때문에 안전하게 처리
try:
    df_train = pd.read_csv(TRAIN_PATH)
except:
    df_train = pd.read_table(TRAIN_PATH, sep=";")

df_train["comments"] = df_train["comments"].astype(str)
df_train["label"] = df_train["label"].astype(str)

def _norm_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.strip()
    s = re.sub(r"\s+", "", s)
    return s

def _ngrams3(s: str):
    s = _norm_text(s)
    if len(s) < 3:
        return []
    return [s[i:i+3] for i in range(len(s)-2)]

# inverted index: 3-gram -> [doc_ids...]
inv = defaultdict(list)
train_comments_norm = []
train_labels = df_train["label"].tolist()

for idx, txt in enumerate(df_train["comments"].tolist()):
    tn = _norm_text(txt)
    train_comments_norm.append(tn)
    grams = set(_ngrams3(tn))
    for g in grams:
        inv[g].append(idx)

def retrieve_examples_for_chunk(chunk_comments, top_k=5, max_candidates=5000):
    """
    chunk 내 댓글들을 합쳐서 query를 만들고,
    train에서 3-gram 겹침이 많은 예시 top_k를 뽑아 반환.
    """
    # chunk에서 너무 길면 토큰 폭발하므로 일부만 사용 (최소 수정 + 안전장치)
    sample = chunk_comments[:20]  # 필요 시 10~30 조정
    query = _norm_text(" ".join(sample))
    q_grams = set(_ngrams3(query))
    if not q_grams:
        return []

    cand = []
    for g in q_grams:
        cand.extend(inv.get(g, []))

    if not cand:
        return []

    # 후보가 너무 많으면 상위 빈도 후보만 줄임
    cnt = Counter(cand)
    cand_ids = [doc_id for doc_id, _ in cnt.most_common(max_candidates)]

    # 점수 = 공유 3-gram 수
    scored = []
    q_grams_set = q_grams
    for doc_id in cand_ids:
        d_grams = set(_ngrams3(train_comments_norm[doc_id]))
        score = len(q_grams_set & d_grams)
        if score > 0:
            scored.append((score, doc_id))

    scored.sort(reverse=True, key=lambda x: x[0])
    top = scored[:top_k]

    out = []
    for score, doc_id in top:
        cmt = df_train.iloc[doc_id]["comments"]
        lab = df_train.iloc[doc_id]["label"]
        out.append((str(cmt), str(lab)))
    return out


def get_labeling(model, chunk, max_retries=10, sleep_sec=1):
    print('Model: ', model)
    dfs = []
    failed_chunks = []  # 1-based chunk 번호 저장(로그와 동일)

    for i, chunk in enumerate(chunks):
        success = False

        # ============================================================
        # ✅ [ADD 2] 이 청크에 대해 Retrieval 수행 (RAG 핵심)
        # ============================================================
        chunk_list = chunk.to_list()
        retrieved = retrieve_examples_for_chunk(chunk_list, top_k=5)

        retrieved_block = ""
        if retrieved:
            lines = []
            for ex_c, ex_l in retrieved:
                ex_c_short = str(ex_c).replace("\n", " ").strip()
                ex_c_short = ex_c_short[:80]  # 프롬프트 과대 방지
                # 라벨 공백 제거 (혹시 "3, 5" 같은 거 대비)
                ex_l = str(ex_l).replace(" ", "")
                lines.append(f"- comment: {ex_c_short} | label: {ex_l}")
            retrieved_block = "\n".join(lines)
        else:
            retrieved_block = "- (no retrieved examples)"

        for attempt in range(1, max_retries + 1):
            try:
                response = client.models.generate_content(
                    model=model, contents=[
                        f"""

                        ==========================================================================================

                        너의 업무는 KOR_hatespeech_all_test 파일의 모든 comments에 대해서 혐오 표현을 구별하여 주어진 9가지 카테고리에 맞춰 라벨링을 하는 거야.
                        KOR_hatespeech_all_test 있는 코멘트가 주어진 9개의 카테고리 중 어디에 해당하는지 예측한 결과를 나타내줘.
                        예측할 때 KOR_hatespeech_all_samlping 파일에서 유사한 문서를 k개 찾아서(k=5) 그것을 근거로 하여 업무를 수행해줘.
                        각각의 카테고리는 숫자로 표현되고, 0번부터 7번에 해당하는 혐오표현의 경우 다중으로 라벨링 될 수 있어.
                        KOR_hatespeech_all_test 파일은 코멘트 열만 있는데, 그 코멘트 열 옆에 각 카테고리를 예측한 label이라는 열을 추가하여 csv 파일로 반환해줘.
                        label 열에 들어가는 값은 문자열로만 가능하고, 만약 다중으로 라벨링이 될 경우 다른 기호 없이 숫자와 쉼표로만 나타내줘(예: 다중 레이블의 경우 "3,5"로 오름차순으로 표현).

                        ### 카테고리 정의
                        0 출신차별(origin): 출신 지역, 고향 등을 모욕 또는 비난하거나 배제하고 차별을 정당화하는 경우. 출신 국가, 지역, 국적, 이민 상태에 기반한 멸칭 등을 포함한 혐오 발언
                        1 외모차별(physical): 신체적 특징, 장애로 조롱하고 비난하거나 이로 인해 개인의 가치나 능력을 낮게 평가하고 비난하는 발언
                        2 정치성향차별(political): 특정 정당 지지, 이념적 견해, 정치 성향 등을 이유로 비난하고 이를 정당화하는 발언
                        3 혐오욕설(profanitiy): 일반적인 욕설, 특정한 개인에 대한 비속어 형태의 비난이 포함되어 있는 발언
                        4 연령차별(age): 나이 또는 세대로 집단을 비하하고 조롱하는 발언
                        5 성차별(gender): 성별, 개인의 성 지향성을 기반으로 편견을 가지고 차별하고 비난하는 발언
                        6 인종차별(race): 피부색, 인종, 민족적 배경 등을 이유로 개인이나 집단을 모욕하는 경우. 인종, 피부색, 민족 혈통 자체에 기반한 혐오 발언
                        7 종교차별(religion): 특정 종교에 대한 편견으로 개인이나 집단을 모욕하는 발언
                        8 해당사항없음(not hate speech): 위 기준에 해당되지 않음. 혐오발언 없음.

                        ===========================================================================================

                        [Retrieved examples from train set]
                        {retrieved_block}

                        ===========================================================================================

                        Rules:
                        - DO NOT write code.
                        - DO NOT define functions.
                        - DO NOT explain.
                        - Return ONLY a Python dictionary literal.
                        - Length of 'comments' and 'label' MUST be exactly {len(chunk_list)}
                        - "8"이면 다른 번호와 중복 라벨링하지 말고 반드시 "8" 단독

                        input dataset
                        - {chunk_list}

                        output format
                        {{
                            'comments': ["...", "...", "..."],
                            'label': ["...", "...","..."]
                        }}

                        """]
                )

                csv_text = response.text.strip()
                if csv_text.startswith("```"):
                    csv_text = "\n".join(csv_text.splitlines()[1:-1])

                data = parse_llm_dict(csv_text)

                df_chunk = pd.DataFrame({
                    "comments": data["comments"],
                    "label": data["label"]
                })

                # ============================================================
                # ✅ [ADD 3] 안전장치(최소 수정): 입력과 comments 길이/내용 불일치면 재시도
                # ============================================================
                if len(df_chunk) != len(chunk_list):
                    raise ValueError(f"len mismatch: got={len(df_chunk)} expected={len(chunk_list)}")

                dfs.append(df_chunk)
                success = True

                print(f"✓ chunk {i+1}/{len(chunks)} done")
                break

            except Exception as e:
                print(f"✗ chunk {i+1}/{len(chunks)} failed: {e}")
                time.sleep(sleep_sec)

        if not success:
            print(f"✗ chunk {i+1} permanently failed → skipped")
            failed_chunks.append(i+1)

    df_final = pd.concat(dfs, ignore_index=True)
    print(df_final)

    df_final.to_csv(f"/{model}_RAG.csv",
                    index=False, index_label=False)

    print("\n===== FAILED CHUNKS (1-based) =====")
    print(failed_chunks)

    with open(f"/{model}_failed_chunks.txt", "w") as f:
        f.write(",".join(map(str, failed_chunks)))

    return failed_chunks



# =========================
# Test 파일 로드 (기존 로직 유지)
# =========================
fileName = "/KOR_hatespeech_all_test.csv"

try:
    df = pd.read_csv(fileName)
except:
    print('read ";" seperator.....!')
    df = pd.read_table(fileName, sep=";")


CHUNK_SIZE = 100
chunks = [
    df.iloc[i:i+CHUNK_SIZE, 0]
    for i in range(0, len(df), CHUNK_SIZE)
]

model = "gemini-2.5-flash"
get_labeling(model=model, chunk=chunks)


read ";" seperator.....!
Model:  gemini-2.5-flash
✗ chunk 1/231 failed: 500 INTERNAL. {'error': {'code': 500, 'message': 'An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting', 'status': 'INTERNAL'}}
✓ chunk 1/231 done
✓ chunk 2/231 done
✓ chunk 3/231 done
✓ chunk 4/231 done
✓ chunk 5/231 done
✓ chunk 6/231 done
✓ chunk 7/231 done
✓ chunk 8/231 done
✓ chunk 9/231 done
✓ chunk 10/231 done
✓ chunk 11/231 done
✓ chunk 12/231 done
✓ chunk 13/231 done
✗ chunk 14/231 failed: All arrays must be of the same length
✓ chunk 14/231 done
✓ chunk 15/231 done
✓ chunk 16/231 done
✓ chunk 17/231 done
✗ chunk 18/231 failed: All arrays must be of the same length
✗ chunk 18/231 failed: All arrays must be of the same length
✓ chunk 18/231 done
✓ chunk 19/231 done
✓ chunk 20/231 done
✓ chunk 21/231 done
✓ chunk 22/231 done
✓ chunk 23/231 done
✓ chunk 24/231 done
✗ chunk 25/231 failed: All arrays must be of the same length
✗ chunk 25/231 fail

[127]

In [ ]:
# 청크실패 - chunk 단위

import ast
import time
import pandas as pd
from google import genai

client = genai.Client(api_key="YOUR_API_KEY")

def parse_llm_dict(text: str):
    text = text.strip()
    if text.startswith("```"):
        text = "\n".join(text.splitlines()[1:-1]).strip()
    return ast.literal_eval(text)

def relabel_one_chunk_to_csv(
    model: str,
    fileName: str,
    chunk_no: int,          # 1-based (예: 210)
    chunk_size: int = 100,
    max_retries: int = 10,
    sleep_sec: int = 2,
    out_csv_path: str = None
):
    # 1) 원본 읽기 (쉼표/따옴표 때문에 ; 권장)
    df = pd.read_csv(fileName, sep=";", engine="python")
    df = df.drop(columns=["Unnamed: 1"], errors="ignore")

    # 2) 청크 범위 계산
    start = (chunk_no - 1) * chunk_size
    end = min(chunk_no * chunk_size, len(df))

    if start >= len(df):
        raise ValueError(f"chunk_no={chunk_no}가 데이터 길이(len={len(df)})를 초과합니다.")

    # 3) 댓글 100개 뽑기 (당신 코드처럼 0번째 컬럼 기준)
    chunk = df.iloc[start:end, 0]  # Series
    comments_list = chunk.to_list()
    n = len(comments_list)

    # 4) 프롬프트 (f-string 쓰므로, 프롬프트 안의 { } 예시는 {{ }}로)
    prompt = f"""

                        너의 업무는 KOR_hatespeech_all_test 파일의 모든 comments에 대해서 혐오 표현을 구별하여 주어진 9가지 카테고리에 맞춰 라벨링을 하는 거야.
                        KOR_hatespeech_all_test 있는 코멘트가 주어진 9개의 카테고리 중 어디에 해당하는지 예측한 결과를 나타내줘.
                        예측할 때 KOR_hatespeech_all_samlping 파일에서 유사한 문서를 k개 찾아서(k=5) 그것을 근거로 하여 업무를 수행해줘.
                        각각의 카테고리는 숫자로 표현되고, 0번부터 7번에 해당하는 혐오표현의 경우 다중으로 라벨링 될 수 있어.
                        KOR_hatespeech_all_test 파일은 코멘트 열만 있는데, 그 코멘트 열 옆에 각 카테고리를 예측한 label이라는 열을 추가하여 csv 파일로 반환해줘.
                        label 열에 들어가는 값은 숫자로만 가능하고, 만약 다중으로 라벨링이 될 경우 다른 기호 없이 숫자와 쉼표로만 나타내줘(예: 다중 레이블의 경우 "3,5"로 오름차순으로 표현).

                        ### 카테고리 정의
                        0 출신차별(origin): 출신 지역, 고향 등을 모욕 또는 비난하거나 배제하고 차별을 정당화하는 경우. 출신 국가, 지역, 국적, 이민 상태에 기반한 멸칭 등을 포함한 혐오 발언
                        1 외모차별(physical): 신체적 특징, 장애로 조롱하고 비난하거나 이로 인해 개인의 가치나 능력을 낮게 평가하고 비난하는 발언
                        2 정치성향차별(political): 특정 정당 지지, 이념적 견해, 정치 성향 등을 이유로 비난하고 이를 정당화하는 발언
                        3 혐오욕설(profanitiy): 일반적인 욕설, 특정한 개인에 대한 비속어 형태의 비난이 포함되어 있는 발언
                        4 연령차별(age): 나이 또는 세대로 집단을 비하하고 조롱하는 발언
                        5 성차별(gender): 성별, 개인의 성 지향성을 기반으로 편견을 가지고 차별하고 비난하는 발언
                        6 인종차별(race): 피부색, 인종, 민족적 배경 등을 이유로 개인이나 집단을 모욕하는 경우. 인종, 피부색, 민족 혈통 자체에 기반한 혐오 발언
                        7 종교차별(religion): 특정 종교에 대한 편견으로 개인이나 집단을 모욕하는 발언
                        8 해당사항없음(not hate speech): 위 기준에 해당되지 않음. 혐오발언 없음.

Rules:
- DO NOT write code.
- DO NOT explain.
- Return ONLY a Python dictionary literal.
- Output must contain ONLY 'label'.
- Length of 'label' MUST be exactly {n}.
- label은 숫자만(다중이면 "3,5"처럼 쉼표만)
- "8"이면 다른 번호와 중복 라벨링하지 말고 반드시 "8" 단독

input dataset:
{comments_list}

출력 형식(예시):
{{'label': ['8', '3', '3,5', '1', '2']}}

output format:
{{'label': ['...', '...', ...]}}
"""

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.models.generate_content(model=model, contents=[prompt])
            data = parse_llm_dict(resp.text)

            labels = data.get("label", None)
            if labels is None:
                raise ValueError("응답에 'label' 키가 없습니다.")
            if not isinstance(labels, list):
                raise ValueError(f"'label'이 list가 아닙니다: {type(labels)}")
            if len(labels) != n:
                raise ValueError(f"label 길이가 다릅니다. got={len(labels)} expected={n}")

            # 5) 청크 결과 DF 만들기 (나중에 끼워넣기 쉽게 row_id 포함)
            out_df = pd.DataFrame({
                "row_id": chunk.index.to_list(),     # 원본에서의 행 위치(0-based)
                "comments": comments_list,
                "label": labels
            })

            # 6) 저장
            if out_csv_path is None:
                out_csv_path = f"/{model}_chunk_{chunk_no}.csv"

            out_df.to_csv(out_csv_path, sep=";", index=False)
            print(f"Saved chunk {chunk_no}: rows {start}~{end-1} -> {out_csv_path}")
            return out_df

        except Exception as e:
            last_err = e
            print(f"[retry {attempt}/{max_retries}] failed:", e)
            time.sleep(sleep_sec)

    raise RuntimeError(f"재시도 {max_retries}회 모두 실패: {last_err}")


# ===== 실행 =====
fileName = "/KOR_hatespeech_all_test.csv"
model = "gemini-2.5-flash"

df127 = relabel_one_chunk_to_csv(
    model=model,
    fileName=fileName,
    chunk_no=127,
    chunk_size=100,
    max_retries=10,
    sleep_sec=2,
    out_csv_path="/chunk_127_labels.csv"
)

df127.head()


Saved chunk 127: rows 12600~12699 -> /content/drive/MyDrive/MP_KORHateSpeechDetection/chunk_127_labels.csv


,row_id,comments,label
0,12600,"씨발 전라도는 끼여야하지만 ""오늘도 홍어가 만선이구나"" ??? 네다틀 네다틀","0,3,4"
1,12601,조선족 색ㅋㅣ들 즐비함!,"0,3"
2,12602,왜 울고 지랄 니나라 가라 왜년아,"0,3,5"
3,12603,천암함 10주기다 이 줭간나 빨갱이들아,"2,3"
4,12604,김제덕 선수 성희롱 하는 남초 댓글 본게 몇개인데 웃기고 앉았ㅋㅋ,5


## 2. GEMINI Baseline

In [ ]:
import ast
import time
import json
import pandas as pd
from google import genai


client = genai.Client(api_key="YOUR_API_KEY")

def parse_llm_dict(text):
    text = text.strip()
    # if text.startswith("{'") or text.startswith("{ \""):
    #     return ast.literal_eval(text)
    # return json.loads(text)
    return ast.literal_eval(text)

def get_labeling(model, chunk,max_retries=10,sleep_sec=1):
    print('Model: ', model)
    dfs = []
    failed_chunks = []  # 1-based chunk 번호 저장(로그와 동일)

    for i, chunk in enumerate(chunks):
        success = False

        for attempt in range(1, max_retries + 1):
            try:
                response = client.models.generate_content(
                    model=model, contents=[
                        f"""

                        ==========================================================================================

                        해당 파일에 있는 코멘트를 확인하고, 각 코멘트들의 카테고리를 예측하여 라벨링을 해줘.
                        파일의 코멘트 열 옆에 각 카테고리를 예측한 label이라는 열을 추가하여 csv 파일로 반환해줘.
                        label 열에 들어가는 값은 숫자만 가능하고, 다중 레이블의 경우 숫자와 쉼표로만 나타내줘.

                        ### 카테고리 목록
                        0 출신차별(origin)
                        1 외모차별(physical)
                        2 정치성향차별(political)
                        3 혐오욕설(profanity)
                        4 연령차별(age)
                        5 성차별(gender)
                        6 인종차별(race)
                        7 종교차별(religion)
                        8 해당사항없음(not hate speech)

                        ===========================================================================================

                        Rules:
                        - DO NOT write code.
                        - DO NOT define functions.
                        - DO NOT explain.
                        - Return ONLY a Python dictionary literal.
                        - Length of 'comments' and 'label' MUST be exactly {len(chunk.to_list())}

                        input dataset
                        - {chunk.to_list()}

                        output format
                        {{
                            'comments': ["...", "...", "..."],
                            'label': ["...", "...","..."]
                        }}

                        """]
                )

                csv_text = response.text.strip()

                # ```csv 제거
                if csv_text.startswith("```"):
                    csv_text = "\n".join(csv_text.splitlines()[1:-1])


                data = parse_llm_dict(csv_text)

                df_chunk = pd.DataFrame({
                    "comments": data["comments"],
                    "label": data["label"]
                })

                dfs.append(df_chunk)
                success = True

                print(f"✓ chunk {i+1}/{len(chunks)} done")
                break  # 성공 시 재시도 루프 탈출

            except Exception as e:
                print(f"✗ chunk {i+1}/{len(chunks)} failed: {e}")
                time.sleep(sleep_sec)

        if not success:
            print(f"✗ chunk {i+1} permanently failed → skipped")
            failed_chunks.append(i+1)


        #TODO: test용. 실제 돌릴때는 주석 처리
        #if i==1: # 1 tries
        #    break

    df_final = pd.concat(dfs, ignore_index=True)

    # print(response_test.text) # 13m 49s
    print(df_final)

    df_final.to_csv(f"/{model}_Vanila.csv", index=False, index_label=False)

    print("\n===== FAILED CHUNKS (1-based) =====")  # ✅ 추가
    print(failed_chunks)

    with open(f"/{model}_failed_chunks.txt", "w") as f:  # ✅ 추가
        f.write(",".join(map(str, failed_chunks)))

    return failed_chunks  # ✅ 선택(추후 재실행에 편함)



##################################################################################################################################################################

##################################################################################################################################################################





# test_dataset = client.files.upload(file="/KOR_hatespeech_all_test.csv")
#df = pd.read_csv("/KOR_hatespeech_all_test.csv")
fileName = "/KOR_hatespeech_all_test.csv"

try:
    df = pd.read_csv(fileName)
except:
    print('read ";" seperator.....!')
    df = pd.read_table(fileName, sep=";")


# TODO: CHUNK_SIZE를 설정해서 주석 풀고 실행!!
CHUNK_SIZE = 100  # ⭐ 매우 중요 (30~100 권장)
chunks = [
    df.iloc[i:i+CHUNK_SIZE,0]
    for i in range(0, len(df), CHUNK_SIZE)
]


model = "gemini-2.5-flash" # ["gemini-3-pro-preview","gemini-3-flash-preview","gemini-2.5-flash","gemini-2.5-pro",'gemma-3-4b-it','gemma-3-12b-it','gemma-3-1b-it'  ]
get_labeling(model=model, chunk=chunks)



read ";" seperator.....!
Model:  gemini-2.5-flash
✗ chunk 1/231 failed: All arrays must be of the same length
✓ chunk 1/231 done
✓ chunk 2/231 done
✓ chunk 3/231 done
✓ chunk 4/231 done
✓ chunk 5/231 done
✗ chunk 6/231 failed: All arrays must be of the same length
✓ chunk 6/231 done
✓ chunk 7/231 done
✓ chunk 8/231 done
✓ chunk 9/231 done
✓ chunk 10/231 done
✓ chunk 11/231 done
✓ chunk 12/231 done
✓ chunk 13/231 done
✓ chunk 14/231 done
✓ chunk 15/231 done
✓ chunk 16/231 done
✗ chunk 17/231 failed: All arrays must be of the same length
✓ chunk 17/231 done
✓ chunk 18/231 done
✓ chunk 19/231 done
✓ chunk 20/231 done
✓ chunk 21/231 done
✓ chunk 22/231 done
✓ chunk 23/231 done
✓ chunk 24/231 done
✓ chunk 25/231 done
✓ chunk 26/231 done
✗ chunk 27/231 failed: 500 INTERNAL. {'error': {'code': 500, 'message': 'An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting', 'status': 'INTERNAL'}}
✓ chunk 27/231 done
✓ chunk 28/231 done
✓

[]

In [ ]:
# 청크실패 재시도 - 개별
import ast
import time
from google import genai


client = genai.Client(api_key="YOUR_API_KEY")


def parse_llm_dict(text: str):
    text = text.strip()
    # 모델이 ```로 감싼 경우 제거
    if text.startswith("```"):
        text = "\n".join(text.splitlines()[1:-1]).strip()
    return ast.literal_eval(text)

def label_only_once(model: str, max_retries=10, sleep_sec=2):
    n = 1

    prompt = f"""
                        ==========================================================================================

                        해당 파일에 있는 코멘트를 확인하고, 각 코멘트들의 카테고리를 예측하여 라벨링을 해줘.
                        파일의 코멘트 열 옆에 각 카테고리를 예측한 label이라는 열을 추가하여 csv 파일로 반환해줘.
                        label 열에 들어가는 값은 숫자만 가능하고, 다중 레이블의 경우 숫자와 쉼표로만 나타내줘.

                        ### 카테고리 목록
                        0 출신차별(origin)
                        1 외모차별(physical)
                        2 정치성향차별(political)
                        3 혐오욕설(profanity)
                        4 연령차별(age)
                        5 성차별(gender)
                        6 인종차별(race)
                        7 종교차별(religion)
                        8 해당사항없음(not hate speech)

                        ===========================================================================================


Rules:
- DO NOT write code.
- DO NOT explain.
- Return ONLY a Python dictionary literal.
- Output must contain ONLY 'label'.
- Length of 'label' MUST be exactly {n}.
- label은 숫자만(다중이면 "3,5"처럼 쉼표만)
- "8"이면 다른 번호와 중복 라벨링하지 말고 반드시 "8" 단독

input dataset:
[
'삼시세끼랑 차이가 뭐지?']

출력 형식(예시):
{{'label': ['2']}}

output format:
{{'label': ['...', '...', ...]}}
"""

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(model=model, contents=[prompt])

            data = parse_llm_dict(response.text)
            labels = data.get("label", None)

            if labels is None:
                raise ValueError("응답에 'label' 키가 없습니다.")
            if not isinstance(labels, list):
                raise ValueError(f"'label'이 list가 아닙니다: {type(labels)}")
            if len(labels) != n:
                raise ValueError(f"label 길이가 다릅니다. got={len(labels)} expected={n}")

            return labels

        except Exception as e:
            last_err = e
            print(f"[retry {attempt}/{max_retries}] failed:", e)
            time.sleep(sleep_sec)

    raise RuntimeError(f"재시도 {max_retries}회 모두 실패: {last_err}")

# ---- 실행(콘솔 출력) ----
model = "gemini-2.5-flash"
labels = label_only_once(model=model, max_retries=10, sleep_sec=2)

print("\n===== labels only =====")
print(labels)

print("\n===== label + comment =====")
comments = [
 '삼시세끼랑 차이가 뭐지?']
for c, l in zip(comments, labels):
    print(l, "|", c)




===== labels only =====
['8']

===== label + comment =====
8 | 삼시세끼랑 차이가 뭐지?


In [ ]:
# 청크실패 - chunk 단위

import ast
import time
import pandas as pd
from google import genai

client = genai.Client(api_key="YOUR_API_KEY")

def parse_llm_dict(text: str):
    text = text.strip()
    if text.startswith("```"):
        text = "\n".join(text.splitlines()[1:-1]).strip()
    return ast.literal_eval(text)

def relabel_one_chunk_to_csv(
    model: str,
    fileName: str,
    chunk_no: int,          # 1-based (예: 210)
    chunk_size: int = 100,
    max_retries: int = 10,
    sleep_sec: int = 2,
    out_csv_path: str = None
):
    # 1) 원본 읽기 (쉼표/따옴표 때문에 ; 권장)
    df = pd.read_csv(fileName, sep=";", engine="python")
    df = df.drop(columns=["Unnamed: 1"], errors="ignore")

    # 2) 청크 범위 계산
    start = (chunk_no - 1) * chunk_size
    end = min(chunk_no * chunk_size, len(df))

    if start >= len(df):
        raise ValueError(f"chunk_no={chunk_no}가 데이터 길이(len={len(df)})를 초과합니다.")

    # 3) 댓글 100개 뽑기 (당신 코드처럼 0번째 컬럼 기준)
    chunk = df.iloc[start:end, 0]  # Series
    comments_list = chunk.to_list()
    n = len(comments_list)

    # 4) 프롬프트 (f-string 쓰므로, 프롬프트 안의 { } 예시는 {{ }}로)
    prompt = f"""
                        ==========================================================================================

                        해당 파일에 있는 코멘트를 확인하고, 각 코멘트들의 카테고리를 예측하여 라벨링을 해줘.
                        파일의 코멘트 열 옆에 각 카테고리를 예측한 label이라는 열을 추가하여 csv 파일로 반환해줘.
                        label 열에 들어가는 값은 숫자만 가능하고, 다중 레이블의 경우 숫자와 쉼표로만 나타내줘.

                        ### 카테고리 목록
                        0 출신차별(origin)
                        1 외모차별(physical)
                        2 정치성향차별(political)
                        3 혐오욕설(profanity)
                        4 연령차별(age)
                        5 성차별(gender)
                        6 인종차별(race)
                        7 종교차별(religion)
                        8 해당사항없음(not hate speech)

                        ===========================================================================================

                        Rules:
                        - DO NOT write code.
                        - DO NOT define functions.
                        - DO NOT explain.
                        - Return ONLY a Python dictionary literal.
                        - Length of 'comments' and 'label' MUST be exactly {len(chunk.to_list())}

input dataset:
{comments_list}

출력 형식(예시):
{{'label': ['8', '3', '3,5', '1', '2']}}

output format:
{{'label': ['...', '...', ...]}}
"""

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.models.generate_content(model=model, contents=[prompt])
            data = parse_llm_dict(resp.text)

            labels = data.get("label", None)
            if labels is None:
                raise ValueError("응답에 'label' 키가 없습니다.")
            if not isinstance(labels, list):
                raise ValueError(f"'label'이 list가 아닙니다: {type(labels)}")
            if len(labels) != n:
                raise ValueError(f"label 길이가 다릅니다. got={len(labels)} expected={n}")

            # 5) 청크 결과 DF 만들기 (나중에 끼워넣기 쉽게 row_id 포함)
            out_df = pd.DataFrame({
                "row_id": chunk.index.to_list(),     # 원본에서의 행 위치(0-based)
                "comments": comments_list,
                "label": labels
            })

            # 6) 저장
            if out_csv_path is None:
                out_csv_path = f"/{model}_chunk_{chunk_no}.csv"

            out_df.to_csv(out_csv_path, sep=";", index=False)
            print(f"Saved chunk {chunk_no}: rows {start}~{end-1} -> {out_csv_path}")
            return out_df

        except Exception as e:
            last_err = e
            print(f"[retry {attempt}/{max_retries}] failed:", e)
            time.sleep(sleep_sec)

    raise RuntimeError(f"재시도 {max_retries}회 모두 실패: {last_err}")


# ===== 실행 =====
fileName = "/KOR_hatespeech_all_test.csv"
model = "gemini-2.5-flash"

df210 = relabel_one_chunk_to_csv(
    model=model,
    fileName=fileName,
    chunk_no=210,
    chunk_size=100,
    max_retries=10,
    sleep_sec=2,
    out_csv_path="/chunk_170_labels.csv"
)

df210.head()


[retry 1/10] failed: label 길이가 다릅니다. got=99 expected=100
Saved chunk 210: rows 20900~20999 -> /content/drive/MyDrive/MP_KORHateSpeechDetection/chunk_170_labels.csv


,row_id,comments,label
0,20900,소수의 시끄러운 소리를 다수인 것마냥 퍼뜨리며 갈등 조장하는 언론들도 문제,8
1,20901,올해는 유난히 일부 여자들이 튀면서 나라를시끄럽게 하네,5
2,20902,에휴 저머리로 공무원...,3
3,20903,크 30대돼지년들 열폭보소ㅋㅋㅋㅋ니들이 능력없어서 남편한테 빌붙어사는걸 왜여기서 하...,"1,3,4,5"
4,20904,내가아는좌파들 정말 99프로 문란하게산다 겉으론 정의롭고 당당한척 뒤에선 더럽게놀고,2


# 3. GEMINI CoT

In [ ]:
import ast
import time
import json
import pandas as pd
from google import genai


client = genai.Client(api_key="YOUR_API_KEY")

def parse_llm_dict(text):
    text = text.strip()
    # if text.startswith("{'") or text.startswith("{ \""):
    #     return ast.literal_eval(text)
    # return json.loads(text)
    return ast.literal_eval(text)

def get_labeling(model, chunk,max_retries=10,sleep_sec=1):
    print('Model: ', model)
    dfs = []
    failed_chunks = []  # 1-based chunk 번호 저장(로그와 동일)

    for i, chunk in enumerate(chunks):
        success = False

        for attempt in range(1, max_retries + 1):
            try:
                response = client.models.generate_content(
                    model=model, contents=[
                        f"""

                        ==========================================================================================


                        너의 업무는 KOR_hatespeech_all_test 파일에 있는 코멘트들을 아래 9개 카테고리(0~8) 중 해당하는 것으로 분류하는 거야. 해당 파일의 comments행을 제외한 모든 행의 옆에 label 열을 붙여 csv형태로 반환해줘.
                        판단은 다음 4단계를 내부적으로 거쳐 수행해.
                        Step 1. 댓글의 핵심 의미를 파악
                        Step 2. 각 카테고리 정의에 해당하는 표현(표적, 멸칭, 일반화, 조롱, 비난 근거)이 있는지 검토
                        Step 3. 0~7은 동시에 여러 개에 해당할 수 있음을 고려
                        Step 4. 최종적으로 해당하는 모든 카테고리를 선택


                        ### 중요 규칙(반드시 준수):
                        *label은 숫자만 사용
                        *다중 라벨이면 오름차순으로 숫자와 쉼표만 사용(예: 3,5)
                        *어떤 카테고리에도 명확히 해당하지 않으면 8을 출력

                        #### 카테고리 정의
                        0 출신차별(origin): 출신 지역, 고향 등을 모욕 또는 비난하거나 배제하고 차별을 정당화하는 경우. 출신 국가, 지역, 국적, 이민 상태에 기반한 멸칭 등을 포함한 혐오 발언
                        1 외모차별(physical): 신체적 특징, 장애로 조롱하고 비난하거나 이로 인해 개인의 가치나 능력을 낮게 평가하고 비난하는 발언
                        2 정치성향차별(political): 특정 정당 지지, 이념적 견해, 정치 성향 등을 이유로 비난하고 이를 정당화하는 발언
                        3 혐오욕설(profanitiy): 일반적인 욕설, 특정한 개인에 대한 비속어 형태의 비난이 포함되어 있는 발언
                        4 연령차별(age): 나이 또는 세대로 집단을 비하하고 조롱하는 발언
                        5 성차별(gender): 성별, 개인의 성 지향성을 기반으로 편견을 가지고 차별하고 비난하는 발언
                        6 인종차별(race): 피부색, 인종, 민족적 배경 등을 이유로 개인이나 집단을 모욕하는 경우. 인종, 피부색, 민족 혈통 자체에 기반한 혐오 발언
                        7 종교차별(religion): 특정 종교에 대한 편견으로 개인이나 집단을 모욕하는 발언
                        8 해당사항없음(not hate speech): 위 기준에 해당되지 않음. 혐오발언 없음.


                        #### 예시
                        여자가 최수종처럼 했으면 왜 여자만 말 높이냐 여자가 종이냐 불쌍하다 이혼해라 남녀차별 방송 접어라하며 난리도 아니었을텐데 남자가 저러니 보기 좋다는둥 잉꼬부부라는 둥 이러니 김치년들을 혐오하지 않을 수 없다.
                        최종 라벨: 3,5


                        ===========================================================================================

                        Rules:
                        - DO NOT write code.
                        - DO NOT define functions.
                        - DO NOT explain.
                        - Return ONLY a Python dictionary literal.
                        - Length of 'comments' and 'label' MUST be exactly {len(chunk.to_list())}

                        input dataset
                        - {chunk.to_list()}

                        output format
                        {{
                            'comments': ["...", "...", "..."],
                            'label': ["...", "...","..."]
                        }}

                        """]
                )

                csv_text = response.text.strip()

                # ```csv 제거
                if csv_text.startswith("```"):
                    csv_text = "\n".join(csv_text.splitlines()[1:-1])


                data = parse_llm_dict(csv_text)

                df_chunk = pd.DataFrame({
                    "comments": data["comments"],
                    "label": data["label"]
                })

                dfs.append(df_chunk)
                success = True

                print(f"✓ chunk {i+1}/{len(chunks)} done")
                break  # 성공 시 재시도 루프 탈출

            except Exception as e:
                print(f"✗ chunk {i+1}/{len(chunks)} failed: {e}")
                time.sleep(sleep_sec)

        if not success:
            print(f"✗ chunk {i+1} permanently failed → skipped")
            failed_chunks.append(i+1)


        #TODO: test용. 실제 돌릴때는 주석 처리
        #if i==1: # 1 tries
        #    break

    df_final = pd.concat(dfs, ignore_index=True)

    # print(response_test.text) # 13m 49s
    print(df_final)

    df_final.to_csv(f"/{model}_CoT.csv", index=False, index_label=False)

    print("\n===== FAILED CHUNKS (1-based) =====")  # ✅ 추가
    print(failed_chunks)

    with open(f"/{model}_failed_chunks.txt", "w") as f:  # ✅ 추가
        f.write(",".join(map(str, failed_chunks)))

    return failed_chunks  # ✅ 선택(추후 재실행에 편함)



##################################################################################################################################################################

##################################################################################################################################################################





# test_dataset = client.files.upload(file="/KOR_hatespeech_all_test.csv")
#df = pd.read_csv("/KOR_hatespeech_all_test.csv")
fileName = "/KOR_hatespeech_all_test.csv"

try:
    df = pd.read_csv(fileName)
except:
    print('read ";" seperator.....!')
    df = pd.read_table(fileName, sep=";")


# TODO: CHUNK_SIZE를 설정해서 주석 풀고 실행!!
CHUNK_SIZE = 100  # ⭐ 매우 중요 (30~100 권장)
chunks = [
    df.iloc[i:i+CHUNK_SIZE,0]
    for i in range(0, len(df), CHUNK_SIZE)
]


model = "gemini-2.5-flash" # ["gemini-3-pro-preview","gemini-3-flash-preview","gemini-2.5-flash","gemini-2.5-pro",'gemma-3-4b-it','gemma-3-12b-it','gemma-3-1b-it'  ]
get_labeling(model=model, chunk=chunks)



read ";" seperator.....!
Model:  gemini-2.5-flash
✗ chunk 1/231 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
✓ chunk 1/231 done
✓ chunk 2/231 done
✗ chunk 3/231 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
✓ chunk 3/231 done
✓ chunk 4/231 done
✓ chunk 5/231 done
✓ chunk 6/231 done
✗ chunk 7/231 failed: All arrays must be of the same length
✗ chunk 7/231 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
✗ chunk 7/231 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
✗ chunk 7/231 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
✗ chunk 7/231 fa

[7, 28, 38]

In [ ]:
# 청크실패 - chunk 단위

import ast
import time
import pandas as pd
from google import genai

client = genai.Client(api_key="YOUR_API_KEY")

def parse_llm_dict(text: str):
    text = text.strip()
    if text.startswith("```"):
        text = "\n".join(text.splitlines()[1:-1]).strip()
    return ast.literal_eval(text)

def relabel_one_chunk_to_csv(
    model: str,
    fileName: str,
    chunk_no: int,          # 1-based (예: 210)
    chunk_size: int = 100,
    max_retries: int = 10,
    sleep_sec: int = 2,
    out_csv_path: str = None
):
    # 1) 원본 읽기 (쉼표/따옴표 때문에 ; 권장)
    df = pd.read_csv(fileName, sep=";", engine="python")
    df = df.drop(columns=["Unnamed: 1"], errors="ignore")

    # 2) 청크 범위 계산
    start = (chunk_no - 1) * chunk_size
    end = min(chunk_no * chunk_size, len(df))

    if start >= len(df):
        raise ValueError(f"chunk_no={chunk_no}가 데이터 길이(len={len(df)})를 초과합니다.")

    # 3) 댓글 100개 뽑기 (당신 코드처럼 0번째 컬럼 기준)
    chunk = df.iloc[start:end, 0]  # Series
    comments_list = chunk.to_list()
    n = len(comments_list)

    # 4) 프롬프트 (f-string 쓰므로, 프롬프트 안의 { } 예시는 {{ }}로)
    prompt = f"""


                        너의 업무는 KOR_hatespeech_all_test 파일에 있는 코멘트들을 아래 9개 카테고리(0~8) 중 해당하는 것으로 분류하는 거야. 해당 파일의 comments행을 제외한 모든 행의 옆에 label 열을 붙여 csv형태로 반환해줘.
                        판단은 다음 4단계를 내부적으로 거쳐 수행해.
                        Step 1. 댓글의 핵심 의미를 파악
                        Step 2. 각 카테고리 정의에 해당하는 표현(표적, 멸칭, 일반화, 조롱, 비난 근거)이 있는지 검토
                        Step 3. 0~7은 동시에 여러 개에 해당할 수 있음을 고려
                        Step 4. 최종적으로 해당하는 모든 카테고리를 선택


                        ### 중요 규칙(반드시 준수):
                        *label은 숫자만 사용
                        *다중 라벨이면 오름차순으로 숫자와 쉼표만 사용(예: 3,5)
                        *어떤 카테고리에도 명확히 해당하지 않으면 8을 출력

                        #### 카테고리 정의
                        0 출신차별(origin): 출신 지역, 고향 등을 모욕 또는 비난하거나 배제하고 차별을 정당화하는 경우. 출신 국가, 지역, 국적, 이민 상태에 기반한 멸칭 등을 포함한 혐오 발언
                        1 외모차별(physical): 신체적 특징, 장애로 조롱하고 비난하거나 이로 인해 개인의 가치나 능력을 낮게 평가하고 비난하는 발언
                        2 정치성향차별(political): 특정 정당 지지, 이념적 견해, 정치 성향 등을 이유로 비난하고 이를 정당화하는 발언
                        3 혐오욕설(profanitiy): 일반적인 욕설, 특정한 개인에 대한 비속어 형태의 비난이 포함되어 있는 발언
                        4 연령차별(age): 나이 또는 세대로 집단을 비하하고 조롱하는 발언
                        5 성차별(gender): 성별, 개인의 성 지향성을 기반으로 편견을 가지고 차별하고 비난하는 발언
                        6 인종차별(race): 피부색, 인종, 민족적 배경 등을 이유로 개인이나 집단을 모욕하는 경우. 인종, 피부색, 민족 혈통 자체에 기반한 혐오 발언
                        7 종교차별(religion): 특정 종교에 대한 편견으로 개인이나 집단을 모욕하는 발언
                        8 해당사항없음(not hate speech): 위 기준에 해당되지 않음. 혐오발언 없음.


                        #### 예시
                        여자가 최수종처럼 했으면 왜 여자만 말 높이냐 여자가 종이냐 불쌍하다 이혼해라 남녀차별 방송 접어라하며 난리도 아니었을텐데 남자가 저러니 보기 좋다는둥 잉꼬부부라는 둥 이러니 김치년들을 혐오하지 않을 수 없다.
                        최종 라벨: 3,5


Rules:
- DO NOT write code.
- DO NOT explain.
- Return ONLY a Python dictionary literal.
- Output must contain ONLY 'label'.
- Length of 'label' MUST be exactly {n}.
- label은 숫자만(다중이면 "3,5"처럼 쉼표만)
- "8"이면 다른 번호와 중복 라벨링하지 말고 반드시 "8" 단독

input dataset:
{comments_list}

출력 형식(예시):
{{'label': ['8', '3', '3,5', '1', '2']}}

output format:
{{'label': ['...', '...', ...]}}
"""

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.models.generate_content(model=model, contents=[prompt])
            data = parse_llm_dict(resp.text)

            labels = data.get("label", None)
            if labels is None:
                raise ValueError("응답에 'label' 키가 없습니다.")
            if not isinstance(labels, list):
                raise ValueError(f"'label'이 list가 아닙니다: {type(labels)}")
            if len(labels) != n:
                raise ValueError(f"label 길이가 다릅니다. got={len(labels)} expected={n}")

            # 5) 청크 결과 DF 만들기 (나중에 끼워넣기 쉽게 row_id 포함)
            out_df = pd.DataFrame({
                "row_id": chunk.index.to_list(),     # 원본에서의 행 위치(0-based)
                "comments": comments_list,
                "label": labels
            })

            # 6) 저장
            if out_csv_path is None:
                out_csv_path = f"/{model}_chunk_{chunk_no}.csv"

            out_df.to_csv(out_csv_path, sep=";", index=False)
            print(f"Saved chunk {chunk_no}: rows {start}~{end-1} -> {out_csv_path}")
            return out_df

        except Exception as e:
            last_err = e
            print(f"[retry {attempt}/{max_retries}] failed:", e)
            time.sleep(sleep_sec)

    raise RuntimeError(f"재시도 {max_retries}회 모두 실패: {last_err}")


# ===== 실행 =====
fileName = "/KOR_hatespeech_all_test.csv"
model = "gemini-2.5-flash"

df38 = relabel_one_chunk_to_csv(
    model=model,
    fileName=fileName,
    chunk_no=38,
    chunk_size=100,
    max_retries=10,
    sleep_sec=2,
    out_csv_path="/chunk_38_labels.csv"
)

df38.head()


[retry 1/10] failed: label 길이가 다릅니다. got=98 expected=100
Saved chunk 38: rows 3700~3799 -> /content/drive/MyDrive/MP_KORHateSpeechDetection/chunk_38_labels.csv


,row_id,comments,label
0,3700,최악의 여성인권기록 ㅋㅋ 근무시간 펑균치도 고려안한 임금차 ㅋㅋ,5
1,3701,메이커마다 캐스팅 하는 법도 달라요 본인이 주력으로 쓰는 릴은 어떤 형태로 던져...,8
2,3702,탈레반보다 더 악랄한 선동이다 . 난민과 난민촌 차이를 구별하지 못하고 있고 이슬람...,7
3,3703,아빠가 메이저 리거 아니어도 인종차별 받지않고 살며 아시아인이라고 특별히 차별 받지...,6
4,3704,정작 오바마때도 흑인인권 안챙겨줌,6


In [ ]:
# 청크실패 재시도 - 개별
import ast
import time
from google import genai


client = genai.Client(api_key="YOUR_API_KEY")


def parse_llm_dict(text: str):
    text = text.strip()
    # 모델이 ```로 감싼 경우 제거
    if text.startswith("```"):
        text = "\n".join(text.splitlines()[1:-1]).strip()
    return ast.literal_eval(text)

def label_only_once(model: str, max_retries=5, sleep_sec=2):
    n = 3

    prompt = f"""
                        너의 업무는 KOR_hatespeech_all_test 파일에 있는 코멘트들을 아래 9개 카테고리(0~8) 중 해당하는 것으로 분류하는 거야. 해당 파일의 comments행을 제외한 모든 행의 옆에 label 열을 붙여 csv형태로 반환해줘.
                        판단은 다음 4단계를 내부적으로 거쳐 수행해.
                        Step 1. 댓글의 핵심 의미를 파악
                        Step 2. 각 카테고리 정의에 해당하는 표현(표적, 멸칭, 일반화, 조롱, 비난 근거)이 있는지 검토
                        Step 3. 0~7은 동시에 여러 개에 해당할 수 있음을 고려
                        Step 4. 최종적으로 해당하는 모든 카테고리를 선택


                        ### 중요 규칙(반드시 준수):
                        *label은 숫자만 사용
                        *다중 라벨이면 오름차순으로 숫자와 쉼표만 사용(예: 3,5)
                        *어떤 카테고리에도 명확히 해당하지 않으면 8을 출력

                        #### 카테고리 정의
                        0 출신차별(origin): 출신 지역, 고향 등을 모욕 또는 비난하거나 배제하고 차별을 정당화하는 경우. 출신 국가, 지역, 국적, 이민 상태에 기반한 멸칭 등을 포함한 혐오 발언
                        1 외모차별(physical): 신체적 특징, 장애로 조롱하고 비난하거나 이로 인해 개인의 가치나 능력을 낮게 평가하고 비난하는 발언
                        2 정치성향차별(political): 특정 정당 지지, 이념적 견해, 정치 성향 등을 이유로 비난하고 이를 정당화하는 발언
                        3 혐오욕설(profanitiy): 일반적인 욕설, 특정한 개인에 대한 비속어 형태의 비난이 포함되어 있는 발언
                        4 연령차별(age): 나이 또는 세대로 집단을 비하하고 조롱하는 발언
                        5 성차별(gender): 성별, 개인의 성 지향성을 기반으로 편견을 가지고 차별하고 비난하는 발언
                        6 인종차별(race): 피부색, 인종, 민족적 배경 등을 이유로 개인이나 집단을 모욕하는 경우. 인종, 피부색, 민족 혈통 자체에 기반한 혐오 발언
                        7 종교차별(religion): 특정 종교에 대한 편견으로 개인이나 집단을 모욕하는 발언
                        8 해당사항없음(not hate speech): 위 기준에 해당되지 않음. 혐오발언 없음.


                        #### 예시
                        여자가 최수종처럼 했으면 왜 여자만 말 높이냐 여자가 종이냐 불쌍하다 이혼해라 남녀차별 방송 접어라하며 난리도 아니었을텐데 남자가 저러니 보기 좋다는둥 잉꼬부부라는 둥 이러니 김치년들을 혐오하지 않을 수 없다.
                        최종 라벨: 3,5


Rules:
- DO NOT write code.
- DO NOT explain.
- Return ONLY a Python dictionary literal.
- Output must contain ONLY 'label'.
- Length of 'label' MUST be exactly {n}.
- label은 숫자만(다중이면 "3,5"처럼 쉼표만)
- "8"이면 다른 번호와 중복 라벨링하지 말고 반드시 "8" 단독

input dataset:
[
 '역시 대 중 소',
 '왜 내가 화나지. 독일 얘기가 아니고 우리나라에서도 곧 적용될 상황같노... 똥남아던 좃선족이던...',
 '베댓분 애가 넷인데 도와줄 친정엄마도 없고여행도 갈수있는 여유있는 사람도 아니고 SNS에이쁜거 올릴수도 없고 누가 낳으라고 등떠밀려서낳아서 유암종판정 받으셨나봐요^^사상이 아주 글러먹었네요 어쩜 저리 베베 꼬이다못해스크류바같은 쿵쾅이메갈같을까',
 '한남들이왈왈왈. 여성에 대한 잔인한 강력범죄는 매일 쏟아지고. 가정이나 사회에서 이등 신분 여성에겐 헬조선'
 '역시 여자의 적은 여자 ,여자가 깉은 동성에게 성폭행을 당했는데도 여성단체며 메갈들 아가마 닫고 숨도 안쉬고 있네? 이게 바로 내로남불 ? 기자회견에 연대발표등 왜 안해? 억울한 무고죄인 남자 연예인들은 실명 다 까발리면서 여자 피의자들은 보호해주냐? 여자가수 G NA 도 성매매해서 걸려도 실명 절대 안 올리더라 ,이게 바로 니들이 원하는 남녀평등 유토피아야?'
]

출력 형식(예시):
{{'label': ['8', '3', '3,5', '0', '2']}}

output format:
{{'label': ['...', '...', ...]}}
"""

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(model=model, contents=[prompt])

            data = parse_llm_dict(response.text)
            labels = data.get("label", None)

            if labels is None:
                raise ValueError("응답에 'label' 키가 없습니다.")
            if not isinstance(labels, list):
                raise ValueError(f"'label'이 list가 아닙니다: {type(labels)}")
            if len(labels) != n:
                raise ValueError(f"label 길이가 다릅니다. got={len(labels)} expected={n}")

            return labels

        except Exception as e:
            last_err = e
            print(f"[retry {attempt}/{max_retries}] failed:", e)
            time.sleep(sleep_sec)

    raise RuntimeError(f"재시도 {max_retries}회 모두 실패: {last_err}")

# ---- 실행(콘솔 출력) ----
model = "gemini-2.5-flash"
labels = label_only_once(model=model, max_retries=10, sleep_sec=2)

print("\n===== labels only =====")
print(labels)

print("\n===== label + comment =====")
comments = [
 '조센징들은 참 피곤하게 산다',
 '비정상인 자를 대통령으로 선택한 결과이다. 중국인,일본인과 같은 미개한 국민들...',
 '설리 힘들게한 한남들 전부 처벌받아라 감옥가',
 '여기는 악역들이 진짜 잘해 넘 잼나게 잘보고있음',
 '왜이리 싫어하냐고 물어 보시는데...싸이에 여후배 자는데 강ㄱ드립하고 어느프로에서 박은지씨한테 텃세 엄청 심하게하고 후배 군기잡는법 이야기 했는데 연예인들 노이해했고 그리고 유명한 버스 노상방뇨사건, 또 시청자비난에 반사드립하셨으며 말도 안되는 여성 우월발언하셨음...그외 후배 몇년째 공개구혼등등'
 '씨발 자지있던 놈들이 자지 민주화시키니까 마인드까지 민주화되네제발 내옆 반경 100미터내에 접근안했으면 좋겠다']
for c, l in zip(comments, labels):
    print(l, "|", c)




===== labels only =====
['8', '8', '3,5']

===== label + comment =====
8 | 안녕하세요 김영삼인 거 같네
8 | 서태지는 그와중에 모르세로 있군
3,5 | 이런걸 말꼬리 잡는 꼴페미들 해병대 보내자.


## 4. GEMINI PERSONA

In [ ]:
import ast
import time
import json
import pandas as pd
from google import genai


client = genai.Client(api_key="YOUR_API_KEY")

def parse_llm_dict(text):
    text = text.strip()
    # if text.startswith("{'") or text.startswith("{ \""):
    #     return ast.literal_eval(text)
    # return json.loads(text)
    return ast.literal_eval(text)

def get_labeling(model, chunk,max_retries=10,sleep_sec=1):
    print('Model: ', model)
    dfs = []
    failed_chunks = []  # 1-based chunk 번호 저장(로그와 동일)

    for i, chunk in enumerate(chunks):
        success = False

        for attempt in range(1, max_retries + 1):
            try:
                response = client.models.generate_content(
                    model=model, contents=[
                        f"""

                        ==========================================================================================
              당신은 한국인이며 언어학 및 콘텐츠 모더레이션 분야의 전문가입니다.
              당신의 주요 역할은 주어진 텍스트를 분석하여, 언어 기반 혐오표현을 정확하게 식별하고,
              분류하며, 그 근거를 설명하는 것입니다.

              당신은 특히 다음 요소에 민감하게 반응해야 합니다:
              - 명시적/암묵적 혐오표현
              - 문화적·맥락적 뉘앙스
              - 비하표현(슬러), 고정관념, 비인간화 표현, 은어/암호화된 표현(코드화 언어)
              - 권력 관계 및 표적 집단(타깃 그룹)

              당신은 중립적이고 분석적이며 비판단적인 태도를 유지해야 합니다.
              당신은 텍스트를 검열하거나 수정/재작성하지 않습니다.
              오직 분석과 분류만 수행합니다.

              목표는 정밀도를 해치지 않으면서도 높은 재현율(누락 최소화)을 달성하는 것입니다.
              확신이 어려운 경우에는 그 불확실성을 명시적으로 설명해야 합니다.

              당신의 업무는 해당 파일(KOR_hatespeech_test)에 포함된 모든 코멘트를 확인한 후 주어진 9가지 카테고리에 맞춰 라벨링을 하는 것입니다. 각각의 카테고리는 숫자로 표현되고,
              8번 카테고리는 단독으로 라벨링되며, 0번부터 7번에 해당하는 혐오표현은 다중으로 라벨링 될 수 있습니다.
              해당 파일은 코멘트 열만 있는데, 그 코멘트 열 옆에 각 카테고리를 예측한 label이라는 열을 추가하여 csv 파일로 반환하세요.
              abel 열에 들어가는 값은 숫자만 가능하고, 만약 다중으로 라벨링이 될 경우 다른 기호 없이 숫자와 쉼표로만 나타내야 합니다(예: 다중 레이블의 경우 3,5로 오름차순으로 표현).
              카테고리의 이름과 정의는 다음과 같습니다.

              ### 카테고리 정의
              0 출신차별(origin): 출신 지역, 고향 등을 모욕 또는 비난하거나 배제하고 차별을 정당화하는 경우. 출신 국가, 지역, 국적, 이민 상태에 기반한 멸칭 등을 포함한 혐오 발언
              1 외모차별(physical): 신체적 특징, 장애로 조롱하고 비난하거나 이로 인해 개인의 가치나 능력을 낮게 평가하고 비난하는 발언
              2 정치성향차별(political): 특정 정당 지지, 이념적 견해, 정치 성향 등을 이유로 비난하고 이를 정당화하는 발언
              3 혐오욕설(profanitiy): 일반적인 욕설, 특정한 개인에 대한 비속어 형태의 비난이 포함되어 있는 발언
              4 연령차별(age): 나이 또는 세대로 집단을 비하하고 조롱하는 발언
              5 성차별(gender): 성별, 개인의 성 지향성을 기반으로 편견을 가지고 차별하고 비난하는 발언
              6 인종차별(race): 피부색, 인종, 민족적 배경 등을 이유로 개인이나 집단을 모욕하는 경우. 인종, 피부색, 민족 혈통 자체에 기반한 혐오 발언
              7 종교차별(religion): 특정 종교에 대한 편견으로 개인이나 집단을 모욕하는 발언
              8 해당사항없음(not hate speech): 위 기준에 해당되지 않음. 혐오발언 없음.


                        ===========================================================================================

                        Rules:
                        - DO NOT write code.
                        - DO NOT define functions.
                        - DO NOT explain.
                        - Return ONLY a Python dictionary literal.
                        - Length of 'comments' and 'label' MUST be exactly {len(chunk.to_list())}

                        input dataset
                        - {chunk.to_list()}

                        output format
                        {{
                            'comments': ["...", "...", "..."],
                            'label': ["...", "...","..."]
                        }}

                        """]
                )

                csv_text = response.text.strip()

                # ```csv 제거
                if csv_text.startswith("```"):
                    csv_text = "\n".join(csv_text.splitlines()[1:-1])


                data = parse_llm_dict(csv_text)

                df_chunk = pd.DataFrame({
                    "comments": data["comments"],
                    "label": data["label"]
                })

                dfs.append(df_chunk)
                success = True

                print(f"✓ chunk {i+1}/{len(chunks)} done")
                break  # 성공 시 재시도 루프 탈출

            except Exception as e:
                print(f"✗ chunk {i+1}/{len(chunks)} failed: {e}")
                time.sleep(sleep_sec)

        if not success:
            print(f"✗ chunk {i+1} permanently failed → skipped")
            failed_chunks.append(i+1)


        #TODO: test용. 실제 돌릴때는 주석 처리
        #if i==1: # 1 tries
        #    break

    df_final = pd.concat(dfs, ignore_index=True)

    # print(response_test.text) # 13m 49s
    print(df_final)

    df_final.to_csv(f"/{model}_Persona.csv", index=False, index_label=False)

    print("\n===== FAILED CHUNKS (1-based) =====")  # ✅ 추가
    print(failed_chunks)

    with open(f"/{model}_Persona_failed_chunks.txt", "w") as f:  # ✅ 추가
        f.write(",".join(map(str, failed_chunks)))

    return failed_chunks  # ✅ 선택(추후 재실행에 편함)



##################################################################################################################################################################

##################################################################################################################################################################





# test_dataset = client.files.upload(file="/KOR_hatespeech_all_test.csv")
#df = pd.read_csv("/KOR_hatespeech_all_test.csv")
fileName = "/KOR_hatespeech_all_test.csv"

try:
    df = pd.read_csv(fileName)
except:
    print('read ";" seperator.....!')
    df = pd.read_table(fileName, sep=";")


# TODO: CHUNK_SIZE를 설정해서 주석 풀고 실행!!
CHUNK_SIZE = 100  # ⭐ 매우 중요 (30~100 권장)
chunks = [
    df.iloc[i:i+CHUNK_SIZE,0]
    for i in range(0, len(df), CHUNK_SIZE)
]


model = "gemini-2.5-flash" # ["gemini-3-pro-preview","gemini-3-flash-preview","gemini-2.5-flash","gemini-2.5-pro",'gemma-3-4b-it','gemma-3-12b-it','gemma-3-1b-it'  ]
get_labeling(model=model, chunk=chunks)



read ";" seperator.....!
Model:  gemini-2.5-flash
✓ chunk 1/231 done
✓ chunk 2/231 done
✓ chunk 3/231 done
✓ chunk 4/231 done
✓ chunk 5/231 done
✓ chunk 6/231 done
✓ chunk 7/231 done
✓ chunk 8/231 done
✓ chunk 9/231 done
✓ chunk 10/231 done
✓ chunk 11/231 done
✓ chunk 12/231 done
✗ chunk 13/231 failed: All arrays must be of the same length
✗ chunk 13/231 failed: All arrays must be of the same length
✗ chunk 13/231 failed: All arrays must be of the same length
✗ chunk 13/231 failed: All arrays must be of the same length
✓ chunk 13/231 done
✓ chunk 14/231 done
✓ chunk 15/231 done
✓ chunk 16/231 done
✓ chunk 17/231 done
✗ chunk 18/231 failed: All arrays must be of the same length
✓ chunk 18/231 done
✓ chunk 19/231 done
✓ chunk 20/231 done
✓ chunk 21/231 done
✓ chunk 22/231 done
✓ chunk 23/231 done
✓ chunk 24/231 done
✓ chunk 25/231 done
✓ chunk 26/231 done
✗ chunk 27/231 failed: All arrays must be of the same length
✓ chunk 27/231 done
✗ chunk 28/231 failed: All arrays must be of the same

[]

In [ ]:
# 청크실패 재시도 - 개별
import ast
import time
from google import genai


client = genai.Client(api_key="YOUR_API_KEY")


def parse_llm_dict(text: str):
    text = text.strip()
    # 모델이 ```로 감싼 경우 제거
    if text.startswith("```"):
        text = "\n".join(text.splitlines()[1:-1]).strip()
    return ast.literal_eval(text)

def label_only_once(model: str, max_retries=10, sleep_sec=2):
    n = 5

    prompt = f"""
                    당신은 한국인이며 언어학 및 콘텐츠 모더레이션 분야의 전문가입니다.
              당신의 주요 역할은 주어진 텍스트를 분석하여, 언어 기반 혐오표현을 정확하게 식별하고,
              분류하며, 그 근거를 설명하는 것입니다.

              당신은 특히 다음 요소에 민감하게 반응해야 합니다:
              - 명시적/암묵적 혐오표현
              - 문화적·맥락적 뉘앙스
              - 비하표현(슬러), 고정관념, 비인간화 표현, 은어/암호화된 표현(코드화 언어)
              - 권력 관계 및 표적 집단(타깃 그룹)

              당신은 중립적이고 분석적이며 비판단적인 태도를 유지해야 합니다.
              당신은 텍스트를 검열하거나 수정/재작성하지 않습니다.
              오직 분석과 분류만 수행합니다.

              목표는 정밀도를 해치지 않으면서도 높은 재현율(누락 최소화)을 달성하는 것입니다.
              확신이 어려운 경우에는 그 불확실성을 명시적으로 설명해야 합니다.

              당신의 업무는 해당 파일(KOR_hatespeech_test)에 포함된 모든 코멘트를 확인한 후 주어진 9가지 카테고리에 맞춰 라벨링을 하는 것입니다. 각각의 카테고리는 숫자로 표현되고,
              8번 카테고리는 단독으로 라벨링되며, 0번부터 7번에 해당하는 혐오표현은 다중으로 라벨링 될 수 있습니다.
              해당 파일은 코멘트 열만 있는데, 그 코멘트 열 옆에 각 카테고리를 예측한 label이라는 열을 추가하여 csv 파일로 반환하세요.
              abel 열에 들어가는 값은 숫자만 가능하고, 만약 다중으로 라벨링이 될 경우 다른 기호 없이 숫자와 쉼표로만 나타내야 합니다(예: 다중 레이블의 경우 3,5로 오름차순으로 표현).
              카테고리의 이름과 정의는 다음과 같습니다.

              ### 카테고리 정의
              0 출신차별(origin): 출신 지역, 고향 등을 모욕 또는 비난하거나 배제하고 차별을 정당화하는 경우. 출신 국가, 지역, 국적, 이민 상태에 기반한 멸칭 등을 포함한 혐오 발언
              1 외모차별(physical): 신체적 특징, 장애로 조롱하고 비난하거나 이로 인해 개인의 가치나 능력을 낮게 평가하고 비난하는 발언
              2 정치성향차별(political): 특정 정당 지지, 이념적 견해, 정치 성향 등을 이유로 비난하고 이를 정당화하는 발언
              3 혐오욕설(profanitiy): 일반적인 욕설, 특정한 개인에 대한 비속어 형태의 비난이 포함되어 있는 발언
              4 연령차별(age): 나이 또는 세대로 집단을 비하하고 조롱하는 발언
              5 성차별(gender): 성별, 개인의 성 지향성을 기반으로 편견을 가지고 차별하고 비난하는 발언
              6 인종차별(race): 피부색, 인종, 민족적 배경 등을 이유로 개인이나 집단을 모욕하는 경우. 인종, 피부색, 민족 혈통 자체에 기반한 혐오 발언
              7 종교차별(religion): 특정 종교에 대한 편견으로 개인이나 집단을 모욕하는 발언
              8 해당사항없음(not hate speech): 위 기준에 해당되지 않음. 혐오발언 없음.


Rules:
- DO NOT write code.
- DO NOT explain.
- Return ONLY a Python dictionary literal.
- Output must contain ONLY 'label'.
- Length of 'label' MUST be exactly {n}.
- label은 숫자만(다중이면 "3,5"처럼 쉼표만)
- "8"이면 다른 번호와 중복 라벨링하지 말고 반드시 "8" 단독

input dataset:
[
 '역시 대 중 소',
 '왜 내가 화나지. 독일 얘기가 아니고 우리나라에서도 곧 적용될 상황같노... 똥남아던 좃선족이던...',
 '베댓분 애가 넷인데 도와줄 친정엄마도 없고여행도 갈수있는 여유있는 사람도 아니고 SNS에이쁜거 올릴수도 없고 누가 낳으라고 등떠밀려서낳아서 유암종판정 받으셨나봐요^^사상이 아주 글러먹었네요 어쩜 저리 베베 꼬이다못해스크류바같은 쿵쾅이메갈같을까',
 '한남들이왈왈왈. 여성에 대한 잔인한 강력범죄는 매일 쏟아지고. 가정이나 사회에서 이등 신분 여성에겐 헬조선'
 '역시 여자의 적은 여자 ,여자가 깉은 동성에게 성폭행을 당했는데도 여성단체며 메갈들 아가마 닫고 숨도 안쉬고 있네? 이게 바로 내로남불 ? 기자회견에 연대발표등 왜 안해? 억울한 무고죄인 남자 연예인들은 실명 다 까발리면서 여자 피의자들은 보호해주냐? 여자가수 G NA 도 성매매해서 걸려도 실명 절대 안 올리더라 ,이게 바로 니들이 원하는 남녀평등 유토피아야?'
]

출력 형식(예시):
{{'label': ['8', '3', '3,5', '0', '2']}}

output format:
{{'label': ['...', '...', ...]}}
"""

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(model=model, contents=[prompt])

            data = parse_llm_dict(response.text)
            labels = data.get("label", None)

            if labels is None:
                raise ValueError("응답에 'label' 키가 없습니다.")
            if not isinstance(labels, list):
                raise ValueError(f"'label'이 list가 아닙니다: {type(labels)}")
            if len(labels) != n:
                raise ValueError(f"label 길이가 다릅니다. got={len(labels)} expected={n}")

            return labels

        except Exception as e:
            last_err = e
            print(f"[retry {attempt}/{max_retries}] failed:", e)
            time.sleep(sleep_sec)

    raise RuntimeError(f"재시도 {max_retries}회 모두 실패: {last_err}")

# ---- 실행(콘솔 출력) ----
model = "gemini-2.5-flash"
labels = label_only_once(model=model, max_retries=5, sleep_sec=2)

print("\n===== labels only =====")
print(labels)

print("\n===== label + comment =====")
comments = [
 '역시 대 중 소',
 '왜 내가 화나지. 독일 얘기가 아니고 우리나라에서도 곧 적용될 상황같노... 똥남아던 좃선족이던...',
 '베댓분 애가 넷인데 도와줄 친정엄마도 없고여행도 갈수있는 여유있는 사람도 아니고 SNS에이쁜거 올릴수도 없고 누가 낳으라고 등떠밀려서낳아서 유암종판정 받으셨나봐요^^사상이 아주 글러먹었네요 어쩜 저리 베베 꼬이다못해스크류바같은 쿵쾅이메갈같을까',
 '한남들이왈왈왈. 여성에 대한 잔인한 강력범죄는 매일 쏟아지고. 가정이나 사회에서 이등 신분 여성에겐 헬조선',
 '역시 여자의 적은 여자 ,여자가 깉은 동성에게 성폭행을 당했는데도 여성단체며 메갈들 아가마 닫고 숨도 안쉬고 있네? 이게 바로 내로남불 ? 기자회견에 연대발표등 왜 안해? 억울한 무고죄인 남자 연예인들은 실명 다 까발리면서 여자 피의자들은 보호해주냐? 여자가수 G NA 도 성매매해서 걸려도 실명 절대 안 올리더라 ,이게 바로 니들이 원하는 남녀평등 유토피아야?'
]
for c, l in zip(comments, labels):
    print(l, "|", c)




===== labels only =====
['1,5', '0,3,6', '1,2,3,5', '3,5', '2,3,5']

===== label + comment =====
1,5 | 역시 대 중 소
0,3,6 | 왜 내가 화나지. 독일 얘기가 아니고 우리나라에서도 곧 적용될 상황같노... 똥남아던 좃선족이던...
1,2,3,5 | 베댓분 애가 넷인데 도와줄 친정엄마도 없고여행도 갈수있는 여유있는 사람도 아니고 SNS에이쁜거 올릴수도 없고 누가 낳으라고 등떠밀려서낳아서 유암종판정 받으셨나봐요^^사상이 아주 글러먹었네요 어쩜 저리 베베 꼬이다못해스크류바같은 쿵쾅이메갈같을까
3,5 | 한남들이왈왈왈. 여성에 대한 잔인한 강력범죄는 매일 쏟아지고. 가정이나 사회에서 이등 신분 여성에겐 헬조선
2,3,5 | 역시 여자의 적은 여자 ,여자가 깉은 동성에게 성폭행을 당했는데도 여성단체며 메갈들 아가마 닫고 숨도 안쉬고 있네? 이게 바로 내로남불 ? 기자회견에 연대발표등 왜 안해? 억울한 무고죄인 남자 연예인들은 실명 다 까발리면서 여자 피의자들은 보호해주냐? 여자가수 G NA 도 성매매해서 걸려도 실명 절대 안 올리더라 ,이게 바로 니들이 원하는 남녀평등 유토피아야?
